# Amazon ML Challenge 2026 — CPU training baseline

Train a sampled business-pair classifier against the **full target catalog**, evaluate on untouched S1 groups, save the model, and generate both required outputs.

**Default:** 20,000 sampled S1 entities, 60/20/20 grouped train/calibration/holdout; SQLite BM25 retrieval + incremental logistic regression. No GPU required. This is not the neural pipeline and does not train on every S1 by default. Read [COLAB.md](https://github.com/alisalmann7386-crypto/Amazon-ML-Challange-2026/blob/main/COLAB.md) for limitations.

Put the four training files (TSV or ZIP, exactly one copy each) in Google Drive at `MyDrive/AmazonML2026/input/train`. Large indexing and retrieval runs can take substantial time; start with the default sample before increasing it.


In [ ]:
from pathlib import Path
import subprocess, sys, json, shutil
REPO = Path('/content/Amazon-ML-Challange-2026')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/alisalmann7386-crypto/Amazon-ML-Challange-2026.git',str(REPO)],check=True)
else:
    print('Reusing existing checkout; your local changes are preserved.')
subprocess.run(['git','rev-parse','HEAD'],cwd=REPO,check=True)
VENV = Path('/content/er-venv')
if not (VENV/'bin/python').exists():
    subprocess.run([sys.executable,'-m','venv',str(VENV)],check=True)
PY = str(VENV/'bin/python')
subprocess.run([PY,'-m','pip','install','-r',str(REPO/'requirements.txt')],check=True)
def run(*args):
    subprocess.run([PY,*map(str,args)],cwd=REPO,check=True)
run('-c', "import sqlite3; sqlite3.connect(':memory:').execute('CREATE VIRTUAL TABLE x USING fts5(t)'); print('FTS5 ready')")


## Mount Drive and configure the run
Change `DRIVE_ROOT` to your own folder. Use a new run folder when changing data or retrieval settings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/AmazonML2026')
INPUT_TRAIN = DRIVE_ROOT/'input/train'
INPUT_TEST = DRIVE_ROOT/'input/test'
SAMPLE_SIZE = 20000
K = 20
EPOCHS = 5
SEED = 42
TEAM = 'YOUR_TEAM'  # Replace before packaging.
RUN_TEST = False  # Change to True only after adding all three test files.
DATA = Path('/content/er-data')
CACHE = Path('/content/er-cache')
CACHE.mkdir(parents=True,exist_ok=True)
DRIVE_CACHE = DRIVE_ROOT/'cache'
DRIVE_CACHE.mkdir(parents=True,exist_ok=True)
WORK = DRIVE_ROOT/f'runs/sample{SAMPLE_SIZE}_k{K}_epochs{EPOCHS}_seed{SEED}'
print('Free local disk GB:',round(shutil.disk_usage('/content').free/1e9,2))
print('Outputs and feature checkpoints:',WORK)
run('-m','unittest','discover','-s','tests','-v')


## Prepare training data
This copies only recognized TSVs; raw files stay outside Git. On a resumed session, existing prepared files are reused. Delete the local prepared folder yourself if intentionally changing datasets.

In [ ]:
expected = [f'train_source{i}.tsv' for i in (1,2,3)] + ['train_ground_truth.tsv']
if not all((DATA/'train'/name).exists() for name in expected):
    run('src/prepare_data.py','--input',INPUT_TRAIN,'--output',DATA/'train','--split','train')
print({name:round((DATA/'train'/name).stat().st_size/1e6,1) for name in expected})


## Build the full target index
Runs on CPU and local disk. A completed index is copied to Drive for reuse. The program verifies source hashes, even for a cached index; changing data requires a new index filename.

In [ ]:
TRAIN_INDEX = CACHE/'train.sqlite'
if not TRAIN_INDEX.exists() and (DRIVE_CACHE/'train.sqlite').exists():
    shutil.copy2(DRIVE_CACHE/'train.sqlite',TRAIN_INDEX)
run('src/scalable.py','index','--data',DATA/'train','--split','train','--index',TRAIN_INDEX)
if not (DRIVE_CACHE/'train.sqlite').exists():
    temporary = DRIVE_CACHE/'train.sqlite.copying'
    shutil.copy2(TRAIN_INDEX,temporary)
    temporary.replace(DRIVE_CACHE/'train.sqlite')


## Train, calibrate and evaluate
Completed feature shards are saved on Drive. The threshold uses calibration labels only. The holdout is for evaluation, not for choosing the threshold.

In [ ]:
run('src/scalable.py','train','--index',TRAIN_INDEX,'--work',WORK,
    '--sample-size',SAMPLE_SIZE,'--k',K,'--epochs',EPOCHS,'--seed',SEED)
metrics = json.loads((WORK/'metrics.json').read_text())
print(json.dumps(metrics,indent=2))
print('Saved trained model:',WORK/'model.joblib')
commit = subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
(WORK/'source_commit.txt').write_text(commit+'\n')
with (WORK/'environment.txt').open('w') as f:
    subprocess.run([PY,'-m','pip','freeze'],stdout=f,check=True)


## Optional: predict test data
Set `RUN_TEST=True` above after adding test_source1.tsv, test_source2.tsv and test_source3.tsv under `input/test`. No test labels are used.

In [ ]:
OUTPUT = DRIVE_ROOT/'output'
TEST_INDEX = CACHE/'test.sqlite'
if RUN_TEST:
    expected_test = [f'test_source{i}.tsv' for i in (1,2,3)]
    if not all((DATA/'test'/name).exists() for name in expected_test):
        run('src/prepare_data.py','--input',INPUT_TEST,'--output',DATA/'test','--split','test')
    if not TEST_INDEX.exists() and (DRIVE_CACHE/'test.sqlite').exists():
        shutil.copy2(DRIVE_CACHE/'test.sqlite',TEST_INDEX)
    run('src/scalable.py','index','--data',DATA/'test','--split','test','--index',TEST_INDEX)
    if not (DRIVE_CACHE/'test.sqlite').exists():
        temporary = DRIVE_CACHE/'test.sqlite.copying'
        shutil.copy2(TEST_INDEX,temporary)
        temporary.replace(DRIVE_CACHE/'test.sqlite')
    run('src/scalable.py','predict','--index',TEST_INDEX,'--model',WORK/'model.joblib','--output',OUTPUT)
else:
    print('Test prediction skipped. Model training and holdout evaluation are complete.')


## Optional: package for submission
First fill team details and actual metrics in the repo’s `Documentation_template.md` using Colab’s file editor. This cell does not submit to the competition portal.

In [ ]:
if RUN_TEST:
    if TEAM == 'YOUR_TEAM':
        raise ValueError('Set TEAM before packaging and complete Documentation_template.md.')
    run('src/package_submission.py','--team',TEAM,'--index',TEST_INDEX,
        '--model',WORK/'model.joblib','--output',OUTPUT,'--destination',DRIVE_ROOT/'submissions')
    print('Leaderboard file:',OUTPUT/'matching_results.tsv')
    print('Final archive:',DRIVE_ROOT/'submissions'/f'{TEAM}_submission.zip')
else:
    print('Packaging skipped until test predictions are available.')
